In [ ]:
!nvidia-smi
!pip install -q ultralytics supervision huggingface_hub pandas pyarrow
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!find /content/drive/MyDrive -iname "*.MP4" 2>/dev/null | head

In [ ]:
import os
VID = "/content/drive/MyDrive/Visual Intelligence Hackathon/Intersection_Merged.MP4"  # ← paste from Cell 2
assert os.path.exists(VID), "WRONG PATH — fix before continuing"
print("ok", os.path.getsize(VID)/1e9, "GB")

In [ ]:
%%writefile track.py
import argparse, inspect, time
from pathlib import Path
import cv2, numpy as np, pandas as pd
import supervision as sv
from ultralytics import YOLO
from huggingface_hub import hf_hub_download

VISDRONE={0:"pedestrian",1:"people",2:"bicycle",3:"car",4:"van",5:"truck",
          6:"tricycle",7:"awning-tricycle",8:"bus",9:"motor"}
GROUP={"pedestrian":"pedestrian","people":"pedestrian","bicycle":"cyclist",
       "car":"car","van":"LGV","truck":"HGV","bus":"bus","motor":"motorcycle",
       "tricycle":"three-wheeler","awning-tricycle":"three-wheeler"}

def make_tracker(fps):
    sig=set(inspect.signature(sv.ByteTrack.__init__).parameters); kw={}
    if "frame_rate" in sig: kw["frame_rate"]=int(fps)
    for a,b in [("lost_track_buffer",int(fps*4)),("track_buffer",int(fps*4)),
                ("track_activation_threshold",0.20),("track_thresh",0.20),
                ("minimum_matching_threshold",0.85),("match_thresh",0.85)]:
        if a in sig and a not in kw: kw[a]=b
    print("[tracker]",kw,flush=True)
    return sv.ByteTrack(**kw)

_Q={"m":None}
def predict(model,imgs,device,half,tile,conf):
    if _Q["m"] is None:
        try:
            r=model.predict(imgs,device=device,imgsz=tile,conf=conf,verbose=False,
                            quantize=(16 if half else None))
            _Q["m"]="q"; return r
        except TypeError: _Q["m"]="h"
    if _Q["m"]=="q":
        return model.predict(imgs,device=device,imgsz=tile,conf=conf,verbose=False,
                             quantize=(16 if half else None))
    return model.predict(imgs,device=device,imgsz=tile,conf=conf,verbose=False,half=half)

def tiles(h,w,t,ov):
    s=max(int(t*(1-ov)),1)
    xs=sorted(set(list(range(0,max(w-t,1),s))+[max(w-t,0)]))
    ys=sorted(set(list(range(0,max(h-t,1),s))+[max(h-t,0)]))
    return [(x,y) for y in ys for x in xs]

def detect(model,fr,device,half,t,ov,batch,conf,iou):
    h,w=fr.shape[:2]; co=tiles(h,w,t,ov)
    cr=[fr[y:y+t,x:x+t] for x,y in co]; parts=[]
    for i in range(0,len(cr),batch):
        for r,(x,y) in zip(predict(model,cr[i:i+batch],device,half,t,conf),co[i:i+batch]):
            d=sv.Detections.from_ultralytics(r)
            if len(d):
                d.xyxy=d.xyxy+np.array([x,y,x,y],dtype=np.float32); parts.append(d)
    if not parts: return sv.Detections.empty()
    return sv.Detections.merge(parts).with_nms(threshold=iou,class_agnostic=True)

def frames(cap,s,e,st):
    if s: cap.set(cv2.CAP_PROP_POS_FRAMES,s)
    i=s
    while True:
        if e and i>=e: break
        if not cap.grab(): break
        if (i-s)%st==0:
            ok,f=cap.retrieve()
            if ok: yield i,f
        i+=1

def main():
    p=argparse.ArgumentParser()
    p.add_argument("--video",required=True); p.add_argument("--out-dir",default="/content/out")
    p.add_argument("--start-sec",type=float,default=0); p.add_argument("--max-seconds",type=float,default=0)
    p.add_argument("--stride",type=int,default=3); p.add_argument("--tile",type=int,default=1024)
    p.add_argument("--overlap",type=float,default=0.2); p.add_argument("--batch",type=int,default=12)
    p.add_argument("--conf",type=float,default=0.10); p.add_argument("--nms-iou",type=float,default=0.55)
    p.add_argument("--min-track-len",type=int,default=3); p.add_argument("--device",default="cuda")
    a=p.parse_args()
    out=Path(a.out_dir); out.mkdir(parents=True,exist_ok=True)
    half=a.device.startswith("cuda")
    model=YOLO(hf_hub_download("dronefreak/visdrone-yolov11s","best.pt"))
    cap=cv2.VideoCapture(a.video); assert cap.isOpened(), f"cannot open {a.video}"
    fps=cap.get(cv2.CAP_PROP_FPS) or 29.97
    total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    W=int(cap.get(3)); H=int(cap.get(4))
    s=int(a.start_sec*fps); e=total if a.max_seconds<=0 else min(total,s+int(a.max_seconds*fps))
    n_samp=max((e-s)//a.stride,1)
    print(f"{W}x{H} @{fps:.2f} frames {s}->{e} sampled {n_samp}",flush=True)
    tracker=make_tracker(fps/a.stride)
    rows=[]; t0=time.time(); n=0
    for fi,fr in frames(cap,s,e,a.stride):
        det=tracker.update_with_detections(
            detect(model,fr,a.device,half,a.tile,a.overlap,a.batch,a.conf,a.nms_iou))
        for i in range(len(det)):
            if det.tracker_id is None or det.tracker_id[i] is None: continue
            x1,y1,x2,y2=det.xyxy[i]; cid=int(det.class_id[i]); cn=VISDRONE.get(cid,str(cid))
            rows.append(dict(frame_idx=fi,time_sec=fi/fps,track_id=int(det.tracker_id[i]),
                class_id=cid,class_name=cn,class_group=GROUP.get(cn,cn),
                conf=float(det.confidence[i]),x1=float(x1),y1=float(y1),
                x2=float(x2),y2=float(y2),cx=float((x1+x2)/2),cy_foot=float(y2),
                w=float(x2-x1),h=float(y2-y1),interpolated=False))
        n+=1
        if n%50==0:
            r=n/(time.time()-t0)
            print(f"{n}/{n_samp} {r:.2f} f/s ETA {(n_samp-n)/r/60:.1f} min",flush=True)
    cap.release()
    df=pd.DataFrame(rows)
    if df.empty: print("NO DETECTIONS"); return
    k=df.groupby("track_id").size(); df=df[df.track_id.isin(k[k>=a.min_track_len].index)].copy()
    v=(df.groupby(["track_id","class_group"])["conf"].sum().reset_index()
        .sort_values("conf",ascending=False).drop_duplicates("track_id")
        .set_index("track_id")["class_group"])
    df["class_group"]=df.track_id.map(v)
    df.to_parquet(out/"tracks.parquet",index=False)
    per=df.groupby("frame_idx").size(); L=df.groupby("track_id").size()
    print(f"\n=== RAW QC ===\nrows {len(df)} tracks {df.track_id.nunique()}")
    print(f"objects/frame mean {per.mean():.1f}")
    print(f"track len mean {L.mean():.1f} median {L.median():.0f}")
    print(f"wall {(time.time()-t0)/60:.1f} min")

if __name__=="__main__": main()

In [ ]:
%%writefile stitch.py
import sys, numpy as np, pandas as pd
def stitch(df,fps=29.97,max_gap_sec=2.0,gate_mult=3.0,min_gate=60.0):
    mg=int(max_gap_sec*fps)
    real=df[~df.interpolated] if "interpolated" in df else df
    info={}
    for tid,g in real.groupby("track_id"):
        g=g.sort_values("frame_idx"); n=min(4,len(g))
        def vel(s):
            if len(s)<2: return np.array([0.,0.])
            dt=s.frame_idx.iloc[-1]-s.frame_idx.iloc[0]
            if dt==0: return np.array([0.,0.])
            return np.array([(s.cx.iloc[-1]-s.cx.iloc[0])/dt,
                             (s.cy_foot.iloc[-1]-s.cy_foot.iloc[0])/dt])
        info[tid]=dict(f0=int(g.frame_idx.iloc[0]),f1=int(g.frame_idx.iloc[-1]),
            p0=np.array([g.cx.iloc[0],g.cy_foot.iloc[0]]),
            p1=np.array([g.cx.iloc[-1],g.cy_foot.iloc[-1]]),
            v0=vel(g.iloc[:n]),v1=vel(g.iloc[-n:]),
            size=float(np.sqrt(max(g.w.median()*g.h.median(),1))),cls=g.class_group.iloc[0])
    par={t:t for t in info}
    def find(x):
        while par[x]!=x: par[x]=par[par[x]]; x=par[x]
        return x
    byf0=sorted(info,key=lambda t:info[t]["f0"]); used=set(); pr=0
    for a in sorted(info,key=lambda t:info[t]["f1"]):
        A=info[a]; best=None; bd=1e18
        for b in byf0:
            if b==a or b in used: continue
            B=info[b]; gap=B["f0"]-A["f1"]
            if gap<=0: continue
            if gap>mg: break
            if A["cls"]!=B["cls"]: continue
            d=min(np.linalg.norm(A["p1"]+A["v1"]*gap-B["p0"]),
                  np.linalg.norm(B["p0"]-B["v0"]*gap-A["p1"]))
            gate=max(gate_mult*max(A["size"],B["size"]),min_gate)*(1+gap/mg)
            if d<gate and d<bd: bd,best=d,b
        if best is not None: used.add(best); pr+=1; par[find(best)]=find(a)
    df=df.copy(); df["track_id"]=df.track_id.map(lambda t: find(t) if t in par else t)
    return df,pr
def reinterp(df,stride):
    out=[]
    for tid,g in df.groupby("track_id"):
        g=g.sort_values("frame_idx").drop_duplicates("frame_idx").set_index("frame_idx")
        g=g.reindex(range(int(g.index.min()),int(g.index.max())+1,stride))
        g["interpolated"]=g["cx"].isna()
        c=["x1","y1","x2","y2","cx","cy_foot","w","h","conf"]
        g[c]=g[c].interpolate("linear"); g["track_id"]=tid
        for k in ["class_id","class_name","class_group"]: g[k]=g[k].ffill().bfill()
        out.append(g.reset_index().rename(columns={"index":"frame_idx"}))
    return pd.concat(out,ignore_index=True)
if __name__=="__main__":
    src,dst=sys.argv[1],sys.argv[2]; st=int(sys.argv[3]) if len(sys.argv)>3 else 3
    df=pd.read_parquet(src); print(f"before {df.track_id.nunique()} tracks")
    df,pr=stitch(df); print(f"merged {pr} pairs")
    df=reinterp(df,st); df["time_sec"]=df.frame_idx/29.97
    df.to_parquet(dst,index=False)
    L=df.groupby("track_id").size()
    print(f"after {df.track_id.nunique()} tracks")
    print(f"len mean {L.mean():.1f} median {L.median():.0f} p90 {L.quantile(.9):.0f}")
    print(f"objects/frame mean {df.groupby('frame_idx').size().mean():.1f}")
    print(f"interpolated {100*df.interpolated.mean():.1f}%")
    print(df.groupby('track_id').class_group.first().value_counts())

In [ ]:
%%writefile render.py
import sys, cv2, numpy as np, pandas as pd
C={"car":(0,200,255),"LGV":(0,255,120),"HGV":(255,80,0),"bus":(255,0,200),
   "motorcycle":(80,80,255),"cyclist":(255,255,0),"pedestrian":(0,255,255),
   "three-wheeler":(180,120,255)}
pq,vid,out=sys.argv[1],sys.argv[2],sys.argv[3]
t0=float(sys.argv[4]) if len(sys.argv)>4 else 0
dur=float(sys.argv[5]) if len(sys.argv)>5 else 90
df=pd.read_parquet(pq)
st=int(np.median(np.diff(sorted(df.frame_idx.unique())))) or 3
cap=cv2.VideoCapture(vid); assert cap.isOpened()
fps=cap.get(cv2.CAP_PROP_FPS) or 29.97
W=int(cap.get(3)); H=int(cap.get(4)); OW=1920; OH=int(1920*H/W); sx,sy=OW/W,OH/H
f0=max(int(t0*fps),int(df.frame_idx.min())); f1=min(int((t0+dur)*fps),int(df.frame_idx.max()))
d=df[(df.frame_idx>=f0)&(df.frame_idx<=f1)]
by={k:v for k,v in d.groupby("frame_idx")}
vw=cv2.VideoWriter(out,cv2.VideoWriter_fourcc(*"mp4v"),fps/st,(OW,OH))
cap.set(cv2.CAP_PROP_POS_FRAMES,f0); i=f0; n=0
while i<=f1:
    if not cap.grab(): break
    if i in by:
        ok,fr=cap.retrieve()
        if ok:
            fr=cv2.resize(fr,(OW,OH))
            for r in by[i].itertuples():
                c=C.get(r.class_group,(255,255,255))
                p1=(int(r.x1*sx),int(r.y1*sy)); p2=(int(r.x2*sx),int(r.y2*sy))
                cv2.rectangle(fr,p1,p2,c,2)
                cv2.putText(fr,str(int(r.track_id)),(p1[0],max(p1[1]-4,10)),
                            cv2.FONT_HERSHEY_SIMPLEX,0.45,c,1,cv2.LINE_AA)
            cv2.putText(fr,f"t={i/fps:6.1f}s  n={len(by[i])}",(12,28),
                        cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,255,255),2,cv2.LINE_AA)
            vw.write(fr); n+=1
    i+=1
cap.release(); vw.release()
print(f"wrote {out}: {n} frames, {n*st/fps:.1f}s")

In [ ]:
!python track.py --video "$VID" --out-dir /content/clip90 \
  --start-sec 60 --max-seconds 90 --stride 3 --device cuda

In [ ]:
!python stitch.py /content/clip90/tracks.parquet /content/clip90/tracks_final.parquet 3
!python render.py /content/clip90/tracks_final.parquet "$VID" /content/clip.mp4 60 90
!ffmpeg -y -loglevel error -i /content/clip.mp4 -vcodec libx264 -pix_fmt yuv420p -crf 23 /content/clip_h264.mp4
!mkdir -p /content/drive/MyDrive/flytbase_out
!cp /content/clip90/tracks_final.parquet /content/clip_h264.mp4 /content/drive/MyDrive/flytbase_out/
!ffmpeg -y -loglevel error -i "$VID" -map 0:s:0 /content/drive/MyDrive/flytbase_out/int.srt
!ls -la /content/drive/MyDrive/flytbase_out/

In [ ]:
import cv2, numpy as np
cap=cv2.VideoCapture(VID); cap.set(cv2.CAP_PROP_POS_FRAMES,int(60*29.97))
ok,ref=cap.read(); cap.release(); assert ok
H,W=ref.shape[:2]; print(W,H)
g=ref.copy()
for x in range(0,W,200):
    cv2.line(g,(x,0),(x,H),(0,255,255),1); cv2.putText(g,str(x),(x+3,30),0,0.8,(0,255,255),2)
for y in range(0,H,200):
    cv2.line(g,(0,y),(W,y),(0,255,255),1); cv2.putText(g,str(y),(5,y-5),0,0.8,(0,255,255),2)
cv2.imwrite('/content/grid.png',g)
from IPython.display import Image, display
display(Image('/content/grid.png',width=1400))

In [ ]:
import cv2, numpy as np
from IPython.display import Image, display

cap = cv2.VideoCapture(VID)
cap.set(cv2.CAP_PROP_POS_FRAMES, int(60 * 29.97))
ok, ref = cap.read()
cap.release()
assert ok
H, W = ref.shape[:2]
print(W, H)

# Corridor A — Main arterial (Lower-Left to Upper-Right)
ROAD_A = [
    # Top edge
    (0, 1780), (380, 1680), (620, 1580), (950, 1400), (1240, 1200),
    (1560, 970), (1950, 890), (2080, 780), (2350, 680), (2800, 420),
    (3120, 240), (3360, 80), (3440, 0),
    # Top-right exit
    (3839, 260), (3839, 440),
    # Bottom edge
    (3540, 630), (3080, 980), (2660, 1200), (2430, 1370), (2180, 1440),
    (1850, 1480), (1500, 1620), (1280, 1780), (1130, 1880), (1050, 1920),
    (960, 1960), (740, 2040), (360, 2140), (230, 2159), (0, 2159), (0, 1980)
]

# Corridor B — Cross Road (Top-Left branch + Lower-Right branch)
ROAD_B = [
    # Top-Left Branch (Left edge moving downward, including storefront notch)
    (540, 0), (750, 360), (1020, 730), (1120, 920), (1330, 950), (1380, 1080),

    # Intersection transit
    (1240, 1200), (1850, 1480), (2180, 1440),

    # Lower-Right Branch (Left edge - explicitly shifted right to avoid ALL trees)
    (2350, 1650), (2600, 1900), (2850, 2159),

    # Lower-Right Branch (Right edge - hugging the building footprint back up)
    (3750, 2159), (3400, 1850), (3000, 1500), (2660, 1200),

    # Intersection transit
    (2430, 1370), (1560, 970),

    # Top-Left Branch (Right edge moving upward)
    (1450, 750), (700, 0)
]

POLYS = [ROAD_A, ROAD_B]

mask = np.zeros((H, W), np.uint8)
for p in POLYS:
    cv2.fillPoly(mask, [np.array(p, np.int32)], 255)

# Soften mask boundary
mask = cv2.dilate(mask, np.ones((15, 15), np.uint8))
np.save('/content/road_mask.npy', mask)

# Render overlay check
ov = ref.copy()
ov[mask == 0] = (ov[mask == 0] * 0.25).astype(np.uint8)

for p in POLYS:
    cv2.polylines(ov, [np.array(p, np.int32)], True, (0, 0, 255), 4)

cv2.imwrite('/content/mask_check.png', ov)
display(Image('/content/mask_check.png', width=1400))

In [ ]:
%matplotlib inline
!pip install -q ipympl

In [ ]:
import matplotlib.pyplot as plt, cv2
pts=[]
fig,ax=plt.subplots(figsize=(14,8))
ax.imshow(cv2.cvtColor(ref,cv2.COLOR_BGR2RGB))
def onclick(e):
    if e.xdata is None: return
    pts.append((int(e.xdata),int(e.ydata)))
    ax.plot(e.xdata,e.ydata,'r.',ms=8)
    if len(pts)>1: ax.plot([pts[-2][0],pts[-1][0]],[pts[-2][1],pts[-1][1]],'r-',lw=1)
    fig.canvas.draw_idle()
fig.canvas.mpl_connect('button_press_event',onclick)
plt.title("click around ONE corridor, in order")
plt.show()

In [ ]:
print(pts)   # paste into POLYS

In [ ]:
import numpy as np, cv2, pandas as pd
from IPython.display import Image, display

d=pd.read_parquet('/content/clip90/tracks_final.parquet')
H,W=ref.shape[:2]

# only moving tracks vote — parked cars and reflections don't move
mv=[]
for tid,g in d.groupby('track_id'):
    g=g.sort_values('frame_idx')
    if np.hypot(g.cx.iloc[-1]-g.cx.iloc[0], g.cy_foot.iloc[-1]-g.cy_foot.iloc[0])>150:
        mv.append(g)
mv=pd.concat(mv); print(f"{mv.track_id.nunique()} moving tracks vote on the mask")

acc=np.zeros((H,W),np.uint8)
for tid,g in mv.groupby('track_id'):
    p=np.stack([g.cx,g.cy_foot],1).astype(np.int32)
    cv2.polylines(acc,[p],False,1,thickness=90)     # corridor width in px

mask=((acc>0).astype(np.uint8))*255
mask=cv2.morphologyEx(mask,cv2.MORPH_CLOSE,np.ones((120,120),np.uint8))
mask=cv2.dilate(mask,np.ones((40,40),np.uint8))
np.save('/content/road_mask.npy',mask)

ov=ref.copy(); ov[mask==0]=(ov[mask==0]*0.25).astype(np.uint8)
cv2.imwrite('/content/mask_check.png',ov)
display(Image('/content/mask_check.png',width=1400))

In [ ]:
from google.colab import files
up = files.upload()   # select the makesense .json

In [ ]:
import cv2
cap=cv2.VideoCapture(VID); cap.set(cv2.CAP_PROP_POS_FRAMES,int(60*29.97))
ok,ref=cap.read(); cap.release(); assert ok
H,W=ref.shape[:2]; print("video frame:",W,H)

In [ ]:
import json, numpy as np, cv2
from IPython.display import Image, display

fn=[k for k in up if k.endswith('.json')][0]
j=json.load(open(fn))

POLYS=[]
for v in j.values():
    if not isinstance(v,dict): continue
    regs=v.get('regions',[])
    regs=regs if isinstance(regs,list) else list(regs.values())
    for r in regs:
        sa=r.get('shape_attributes',{})
        if 'all_points_x' in sa:
            POLYS.append(list(zip(sa['all_points_x'],sa['all_points_y'])))
print(f"{len(POLYS)} polygons")

# rescale if annotated image size != video frame size
allx=[p[0] for poly in POLYS for p in poly]; ally=[p[1] for poly in POLYS for p in poly]
print("annotation extent:",max(allx),max(ally))
sx = W/ max(allx) if max(allx) < W*0.85 else 1.0
sy = H/ max(ally) if max(ally) < H*0.85 else 1.0
if sx!=1.0 or sy!=1.0:
    print(f"rescaling by {sx:.3f},{sy:.3f}")
    POLYS=[[(int(x*sx),int(y*sy)) for x,y in poly] for poly in POLYS]

mask=np.zeros((H,W),np.uint8)
for p in POLYS: cv2.fillPoly(mask,[np.array(p,np.int32)],255)
mask=cv2.dilate(mask,np.ones((25,25),np.uint8))
np.save('/content/road_mask.npy',mask)

ov=ref.copy(); ov[mask==0]=(ov[mask==0]*0.25).astype(np.uint8)
for p in POLYS: cv2.polylines(ov,[np.array(p,np.int32)],True,(0,0,255),4)
cv2.imwrite('/content/mask_check.png',ov)
display(Image('/content/mask_check.png',width=1400))

In [ ]:
mask = cv2.bitwise_not(mask)          # annotated regions were NOT road
mask = cv2.erode(mask, np.ones((25,25), np.uint8))   # undo the earlier dilate
np.save('/content/road_mask.npy', mask)

ov = ref.copy(); ov[mask==0] = (ov[mask==0]*0.25).astype(np.uint8)
cv2.imwrite('/content/mask_check.png', ov)
from IPython.display import Image, display
display(Image('/content/mask_check.png', width=1400))

In [ ]:
import pandas as pd, numpy as np
mask=np.load('/content/road_mask.npy')
d=pd.read_parquet('/content/clip90/tracks_final.parquet')
xs=d.cx.clip(0,W-1).astype(int).values; ys=d.cy_foot.clip(0,H-1).astype(int).values
d['in_road']=mask[ys,xs]>0
frac=d.groupby('track_id').in_road.mean()
keep=frac[frac>=0.30].index
before=d.track_id.nunique()
d=d[d.track_id.isin(keep)].copy()
d.to_parquet('/content/clip90/tracks_masked.parquet',index=False)
print(f"tracks {before} -> {d.track_id.nunique()} (dropped {before-d.track_id.nunique()})")
print(d.groupby('track_id').class_group.first().value_counts())

In [ ]:
import numpy as np, pandas as pd, cv2
d=pd.read_parquet('/content/clip90/tracks_masked.parquet')
polys=[np.array(p,np.int32) for p in POLYS]
def edge_dist(x,y):
    return min(abs(cv2.pointPolygonTest(p,(float(x),float(y)),True)) for p in polys)
EDGE=140
rec=[]
for tid,g in d.groupby('track_id'):
    g=g.sort_values('frame_idx')
    rec.append(dict(track_id=tid,cls=g.class_group.iloc[0],n=len(g),dur=len(g)*3/29.97,
        d_in=edge_dist(g.cx.iloc[0],g.cy_foot.iloc[0]),
        d_out=edge_dist(g.cx.iloc[-1],g.cy_foot.iloc[-1]),
        disp=float(np.hypot(g.cx.iloc[-1]-g.cx.iloc[0],g.cy_foot.iloc[-1]-g.cy_foot.iloc[0]))))
r=pd.DataFrame(rec)
r['enters']=r.d_in<EDGE; r['exits']=r.d_out<EDGE
r['complete']=r.enters&r.exits; r['stationary']=r.disp<60
r.to_csv('/content/clip90/track_quality.csv',index=False)
mv=r[~r.stationary]
print(f"tracks {len(r)}  moving {len(mv)}  stationary {r.stationary.sum()}")
print(f"COMPLETE: {mv.complete.mean()*100:.1f}%")
print(f"  starts mid-scene: {(~mv.enters).mean()*100:.1f}%")
print(f"  ends mid-scene:   {(~mv.exits).mean()*100:.1f}%")
print(f"median dur {mv.dur.median():.1f}s  p90 {mv.dur.quantile(.9):.1f}s")
print(mv.groupby('cls').complete.agg(['size','mean']))

In [ ]:
%%writefile trace.py
import sys, cv2, numpy as np, pandas as pd
C={"car":(0,200,255),"LGV":(0,255,120),"HGV":(255,80,0),"bus":(255,0,200),
   "motorcycle":(80,80,255),"cyclist":(255,255,0),"pedestrian":(0,255,255),
   "three-wheeler":(180,120,255)}
pq,vid,mk,out=sys.argv[1],sys.argv[2],sys.argv[3],sys.argv[4]
TAIL=int(sys.argv[5]) if len(sys.argv)>5 else 100
df=pd.read_parquet(pq); mask=np.load(mk)
st=int(np.median(np.diff(sorted(df.frame_idx.unique())))) or 3
cap=cv2.VideoCapture(vid); assert cap.isOpened()
fps=cap.get(cv2.CAP_PROP_FPS) or 29.97
W=int(cap.get(3)); H=int(cap.get(4)); OW=1920; OH=int(1920*H/W); sx,sy=OW/W,OH/H
msk=cv2.resize(mask,(OW,OH))
f0,f1=int(df.frame_idx.min()),int(df.frame_idx.max())
by={k:v for k,v in df.groupby("frame_idx")}; hist={}
vw=cv2.VideoWriter(out,cv2.VideoWriter_fourcc(*"mp4v"),fps/st,(OW,OH))
cap.set(cv2.CAP_PROP_POS_FRAMES,f0); i=f0; n=0
while i<=f1:
    if not cap.grab(): break
    if i in by:
        ok,fr=cap.retrieve()
        if ok:
            fr=cv2.resize(fr,(OW,OH))
            fr[msk==0]=(fr[msk==0]*0.35).astype(np.uint8)
            layer=fr.copy()
            for r in by[i].itertuples():
                p=(int(r.cx*sx),int(r.cy_foot*sy))
                hist.setdefault(r.track_id,[]).append(p)
                pts=hist[r.track_id][-TAIL:]; c=C.get(r.class_group,(255,255,255))
                for k in range(1,len(pts)):
                    a=k/len(pts)
                    cv2.line(layer,pts[k-1],pts[k],tuple(int(v*a) for v in c),
                             max(1,int(1+2*a)),cv2.LINE_AA)
                cv2.circle(fr,p,4,c,-1)
                cv2.putText(fr,str(int(r.track_id)),(p[0]+6,p[1]-6),0,0.45,c,1,cv2.LINE_AA)
            fr=cv2.addWeighted(layer,0.85,fr,0.15,0)
            cv2.putText(fr,f"t={i/fps:6.1f}s active={len(by[i])} tracks={len(hist)}",
                        (12,30),0,0.9,(255,255,255),2,cv2.LINE_AA)
            vw.write(fr); n+=1
    i+=1
cap.release(); vw.release(); print(f"wrote {out}: {n} frames")

In [ ]:
!python trace.py /content/clip90/tracks_masked.parquet "$VID" /content/road_mask.npy /content/trace.mp4 100
!ffmpeg -y -loglevel error -i /content/trace.mp4 -vcodec libx264 -pix_fmt yuv420p -crf 23 /content/trace_h264.mp4

In [ ]:
import cv2, numpy as np, pandas as pd
from IPython.display import Image, display
d=pd.read_parquet('/content/clip90/tracks_masked.parquet')
C={"car":(0,200,255),"LGV":(0,255,120),"HGV":(255,80,0),"bus":(255,0,200),
   "motorcycle":(80,80,255),"cyclist":(255,255,0),"pedestrian":(0,255,255),
   "three-wheeler":(180,120,255)}
canvas=(ref*0.30).astype(np.uint8)
for tid,g in d.groupby('track_id'):
    g=g.sort_values('frame_idx')
    if np.hypot(g.cx.iloc[-1]-g.cx.iloc[0],g.cy_foot.iloc[-1]-g.cy_foot.iloc[0])<60: continue
    pts=np.stack([g.cx,g.cy_foot],1).astype(np.int32)
    cv2.polylines(canvas,[pts],False,C.get(g.class_group.iloc[0],(255,255,255)),2,cv2.LINE_AA)
cv2.imwrite('/content/all_traces.png',canvas)
display(Image('/content/all_traces.png',width=1400))

!mkdir -p /content/drive/MyDrive/flytbase_out
!cp /content/trace_h264.mp4 /content/all_traces.png /content/mask_check.png \
    /content/clip90/tracks_masked.parquet /content/clip90/track_quality.csv \
    /content/drive/MyDrive/flytbase_out/
!ls -la /content/drive/MyDrive/flytbase_out/